In [2]:
from pathlib import Path
import sys
import torch 

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))

from instancepipelineprior import get_data_loaders
from bayesianprior.discretizedtrain import EarthquakeCNN, evaluate_metrics, logits_to_magnitude

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = PROJECT_ROOT / "bayesianprior" / "data" / "best_model_huberandcrosspoint05.pth"

checkpoint= torch.load(checkpoint_path, map_location=device)
model = EarthquakeCNN().to(device)

model.load_state_dict(checkpoint["model_state_dict"])

model.eval()




Train samples: 979487
Validation samples: 96993
Test samples: 82769
Train batches: 7653
Val batches: 758
Test batches: 647
Train samples: 979487
Validation samples: 96993
Test samples: 82769
Train batches: 7653
Val batches: 758
Test batches: 647


EarthquakeCNN(
  (features): Sequential(
    (0): Sequential(
      (0): Conv1d(3, 16, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv1d(16, 32, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (3): Sequential(
      (0): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (4): Sequential(
      (0): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, str

In [3]:
train_loader,val_loader,test_loader = get_data_loaders()


In [4]:
import torch

smoothed_prior = torch.load(
    "data/smoothed_magnitude_prior.pt",
    weights_only=False
)

In [5]:
smoothed_prior = torch.tensor(
    smoothed_prior,
    dtype=torch.float32
)

In [6]:
def logits_to_magnitude_with_prior(logits,bin_centers,prior_probs,alpha):
    prior_probs = prior_probs.to(logits.device)
    bin_centers = bin_centers.to(logits.device)
    log_prior = torch.log(prior_probs.clamp(min=1e-8))

    adjusted_logits = (logits + (alpha - 1.0) * log_prior.unsqueeze(0))
    adjusted_probs = torch.softmax(adjusted_logits,dim=1)
    predictions = torch.sum(adjusted_probs* bin_centers.unsqueeze(0),dim=1)
    return predictions, adjusted_probs

In [7]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

In [8]:
waveforms, magnitudes = next(
    iter(val_loader)
)

waveforms = waveforms.to(device)
magnitudes = magnitudes.to(device)

print(waveforms.shape)
print(magnitudes.shape)

torch.Size([128, 3, 300])
torch.Size([128])


In [9]:
MIN_MAGNITUDE = 0.0
MAX_MAGNITUDE = 6.6
BIN_WIDTH = 0.1

BIN_CENTERS = torch.arange(
    MIN_MAGNITUDE + BIN_WIDTH / 2,
    MAX_MAGNITUDE,
    BIN_WIDTH,
    dtype=torch.float32
)

In [24]:
import math
import torch
import torch.nn.functional as F
def get_normalized_entropy(logits):

    probs = F.softmax(logits, dim=1)

    entropy = -(probs * torch.log(probs + 1e-12)).sum(dim=1)

    entropy = entropy / math.log(probs.shape[1])

    return entropy


def apply_entropy_prior(logits,prior,bin_centers,alpha_max=0.4,power=1.0,magnitude_threshold=4.0):

    probs = F.softmax(logits, dim=1)

    log_model = torch.log(probs + 1e-12)

    prior = prior.to(device=logits.device,dtype=logits.dtype)

    prior = prior / prior.sum()

    log_prior = torch.log(prior + 1e-12).unsqueeze(0)
    entropy = get_normalized_entropy(logits)

    tail_mask = bin_centers >= magnitude_threshold
    tail_probability = probs[:,tail_mask].sum(dim=1)

    alpha = alpha_max * (entropy ** power) * (1 - tail_probability)
    alpha = alpha.unsqueeze(1)

    log_posterior = ((1 - alpha) * log_model+alpha * log_prior)

    posterior = F.softmax(log_posterior,dim=1)

    return (posterior,entropy,alpha.squeeze(1),tail_probability)

In [12]:
logits = model(waveforms)

posterior, entropy, alpha = apply_entropy_prior(logits,smoothed_prior,alpha_max=0.4,power=1.0)

bin_centers_tensor = torch.tensor(BIN_CENTERS,dtype=posterior.dtype,device=posterior.device)

predictions = (posterior * bin_centers_tensor.unsqueeze(0)).sum(dim=1)

probs = posterior

/tmp/ipykernel_6063/3518861193.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  bin_centers_tensor = torch.tensor(BIN_CENTERS,dtype=posterior.dtype,device=posterior.device)


In [ ]:
predictions, probs 

In [ ]:
original_predictions = logits_to_magnitude(
    logits,
    BIN_CENTERS.to(device)
)

print(
    torch.max(
        torch.abs(
            original_predictions - predictions
        )
    )
)

print(
    torch.allclose(
        original_predictions,
        predictions,
        atol=1e-6
    )
)

In [ ]:
import pandas as pd

In [22]:
def evaluate_metrics_with_prior(model, dataloader, device, prior_probs, alpha_max=0.4, power=1.0, magnitude_threshold=4.0):
    model.eval()

    all_predictions = []
    all_targets = []
    all_entropies = []
    all_alphas = []
    all_tail_probabilities = []

    bin_centers = BIN_CENTERS.to(device)

    prior_probs = torch.as_tensor(
        prior_probs,
        dtype=torch.float32,
        device=device
    )

    with torch.no_grad():
        for waveforms, magnitudes in dataloader:
            waveforms = waveforms.to(device, non_blocking=True)
            magnitudes = magnitudes.to(device, non_blocking=True)

            logits = model(waveforms)

            posterior, entropy, alpha, tail_probability = apply_entropy_prior(
                logits,
                prior_probs,
                bin_centers,
                alpha_max=alpha_max,
                power=power,
                magnitude_threshold=magnitude_threshold
            )

            predictions = (
                posterior * bin_centers.unsqueeze(0)
            ).sum(dim=1)

            all_predictions.append(predictions.cpu())
            all_targets.append(magnitudes.cpu())
            all_entropies.append(entropy.cpu())
            all_alphas.append(alpha.cpu())
            all_tail_probabilities.append(tail_probability.cpu())

    all_predictions = torch.cat(all_predictions, dim=0)
    all_targets = torch.cat(all_targets, dim=0)
    all_entropies = torch.cat(all_entropies, dim=0)
    all_alphas = torch.cat(all_alphas, dim=0)
    all_tail_probabilities = torch.cat(all_tail_probabilities, dim=0)

    errors = all_predictions - all_targets

    mae = torch.abs(errors).mean().item()
    rmse = torch.sqrt(torch.mean(errors ** 2)).item()
    bias = errors.mean().item()
    accuracy = (torch.abs(errors) <= 0.2).float().mean().item() * 100

    results = pd.DataFrame({
        "true_magnitude": all_targets.numpy(),
        "predicted_magnitude": all_predictions.numpy(),
        "entropy": all_entropies.numpy(),
        "tail_probability": all_tail_probabilities.numpy(),
        "alpha": all_alphas.numpy()
    })

    return results, mae, rmse, bias, accuracy

In [16]:
def evaluate_metrics_with_prior2(model, dataloader, device, prior_probs, alpha_max=0.4, power=1.0):
    model.eval()

    all_predictions = []
    all_base_predictions = []
    all_targets = []
    all_entropies = []
    all_alphas = []

    bin_centers = BIN_CENTERS.to(device)

    prior_probs = torch.as_tensor(
        prior_probs,
        dtype=torch.float32,
        device=device
    )

    with torch.no_grad():
        for waveforms, magnitudes in dataloader:
            waveforms = waveforms.to(device, non_blocking=True)
            magnitudes = magnitudes.to(device, non_blocking=True)

            logits = model(waveforms)

            # Prediction BEFORE applying prior
            probs = torch.softmax(logits, dim=1)

            base_predictions = (
                probs * bin_centers.unsqueeze(0)
            ).sum(dim=1)

            # Apply entropy-gated prior
            posterior, entropy, alpha = apply_entropy_prior(
                logits,
                prior_probs,
                alpha_max=alpha_max,
                power=power
            )

            # Prediction AFTER applying prior
            predictions = (
                posterior * bin_centers.unsqueeze(0)
            ).sum(dim=1)

            all_predictions.append(predictions.cpu())
            all_base_predictions.append(base_predictions.cpu())
            all_targets.append(magnitudes.cpu())
            all_entropies.append(entropy.cpu())
            all_alphas.append(alpha.cpu())

    all_predictions = torch.cat(all_predictions, dim=0)
    all_base_predictions = torch.cat(all_base_predictions, dim=0)
    all_targets = torch.cat(all_targets, dim=0)
    all_entropies = torch.cat(all_entropies, dim=0)
    all_alphas = torch.cat(all_alphas, dim=0)

    errors = all_predictions - all_targets

    mae = torch.abs(errors).mean().item()
    rmse = torch.sqrt(torch.mean(errors ** 2)).item()
    bias = errors.mean().item()
    accuracy = (torch.abs(errors) <= 0.2).float().mean().item() * 100

    results = pd.DataFrame({
        "true_magnitude": all_targets.numpy(),

        "base_prediction": all_base_predictions.numpy(),
        "prior_prediction": all_predictions.numpy(),

        "entropy": all_entropies.numpy(),
        "alpha": all_alphas.numpy()
    })

    # Error BEFORE prior
    results["base_error"] = (
        results["base_prediction"]
        - results["true_magnitude"]
    )

    results["base_abs_error"] = (
        results["base_error"].abs()
    )

    # Error AFTER prior
    results["prior_error"] = (
        results["prior_prediction"]
        - results["true_magnitude"]
    )

    results["prior_abs_error"] = (
        results["prior_error"].abs()
    )

    return results, mae, rmse, bias, accuracy

In [15]:
import pandas as pd

In [17]:
results, mae, rmse, bias, accuracy = evaluate_metrics_with_prior2(model,val_loader,
    device,
    smoothed_prior,
    alpha_max=0.4,
    power=1.0
)

In [19]:
from scipy.stats import spearmanr

corr, p = spearmanr(
    results["entropy"],
    results["base_abs_error"]
)

print("Spearman correlation:", corr)
print("p-value:", p)

Spearman correlation: 0.33903464687421353
p-value: 0.0


In [20]:
bins = [0, 1, 2, 3, 4, 5, 6]

results["magnitude_bin"] = pd.cut(
    results["true_magnitude"],
    bins=bins,
    right=False
)

magnitude_summary = results.groupby(
    "magnitude_bin",
    observed=True
).agg(
    count=("true_magnitude", "size"),
    mean_entropy=("entropy", "mean"),
    base_mae=("base_abs_error", "mean"),
    base_bias=("base_error", "mean")
)

print(magnitude_summary)

               count  mean_entropy  base_mae  base_bias
magnitude_bin                                          
[0, 1)          5190      0.682420  0.727607   0.727607
[1, 2)         20718      0.672188  0.422286   0.374928
[2, 3)         60450      0.654695  0.275434   0.016304
[3, 4)          9608      0.705847  0.536290  -0.360271
[4, 5)           936      0.735259  0.695998  -0.609000
[5, 6)            91      0.767078  1.015091  -0.985859


In [18]:
results.keys()

Index(['true_magnitude', 'base_prediction', 'prior_prediction', 'entropy',
       'alpha', 'base_error', 'base_abs_error', 'prior_error',
       'prior_abs_error'],
      dtype='object')

In [21]:
results.to_csv("resultsentropy3s.csv")

In [25]:
results2, mae, rmse, bias, accuracy = evaluate_metrics_with_prior(
    model,
    val_loader,
    device,
    smoothed_prior,
    alpha_max=0.4,
    power=1.0,
    magnitude_threshold=4.0
)

In [26]:
print("MAE:", mae)
print("RMSE:", rmse)
print("Bias:", bias)
print("Accuracy ±0.2:", accuracy)

MAE: 0.36415040493011475
RMSE: 0.4876316487789154
Bias: 0.10071167349815369
Accuracy ±0.2: 38.39142918586731


In [28]:
bins = [0, 1, 2, 3, 4, 5, 6]

results2["magnitude_bin"] = pd.cut(
    results2["true_magnitude"],
    bins=bins,
    right=False
)

results2["error"] = (
    results2["predicted_magnitude"]
    - results2["true_magnitude"]
)

results2["abs_error"] = results2["error"].abs()

summary = results2.groupby(
    "magnitude_bin",
    observed=True
).agg(
    count=("true_magnitude", "size"),
    mae=("abs_error", "mean"),
    bias=("error", "mean"),
    mean_entropy=("entropy", "mean"),
    mean_tail_probability=("tail_probability", "mean"),
    mean_alpha=("alpha", "mean")
)

print(summary)

               count       mae      bias  mean_entropy  mean_tail_probability  \
magnitude_bin                                                                   
[0, 1)          5190  0.837370  0.837370      0.682420               0.000983   
[1, 2)         20718  0.475781  0.452594      0.672188               0.001820   
[2, 3)         60450  0.248078  0.016564      0.654695               0.011248   
[3, 4)          9608  0.550655 -0.434024      0.705847               0.098766   
[4, 5)           936  0.773018 -0.725789      0.735259               0.423964   
[5, 6)            91  1.168247 -1.168247      0.767078               0.525048   

               mean_alpha  
magnitude_bin              
[0, 1)           0.272662  
[1, 2)           0.268313  
[2, 3)           0.258486  
[3, 4)           0.252980  
[4, 5)           0.169831  
[5, 6)           0.143204  


In [30]:
print(results2[
    ["entropy", "tail_probability", "alpha"]
].describe())

            entropy  tail_probability         alpha
count  96993.000000      96993.000000  96993.000000
mean       0.665865          0.021819      0.259834
std        0.060258          0.091378      0.029894
min        0.014801          0.000000      0.005920
25%        0.627049          0.000004      0.248041
50%        0.665613          0.000215      0.263310
75%        0.708495          0.003146      0.277159
max        0.910969          0.973763      0.323299


In [33]:
results2["prior_strength_reduction"] = (
    results2["entropy"] * 0.4
    - results2["alpha"]
)

In [34]:
print(
    results2.groupby(
        "magnitude_bin",
        observed=True
    )["prior_strength_reduction"].mean()
)

magnitude_bin
[0, 1)    0.000306
[1, 2)    0.000563
[2, 3)    0.003393
[3, 4)    0.029359
[4, 5)    0.124273
[5, 6)    0.163627
Name: prior_strength_reduction, dtype: float32


In [ ]:
results2.to_csv("detailedentropytail3s.csv")